In [1]:
import cv2
import numpy as np
import json
import os
import glob
import math
from tqdm.notebook import tqdm
from scipy.spatial.distance import euclidean

# --- Configuration ---
IMAGE_DIR = './data/images/'
MASK_DIR = './data/masks/'
CORNER_FILE = 'final_corners.json' # Ensure this matches your saved file
GRAPH_FILE = 'graph_sota.json'
# --- SOTA Parameters ---
NUM_RESAMPLE_POINTS = 50   
FLAT_THRESHOLD = 0.03      
MATCH_THRESHOLD = 15.0     # <--- CHANGED FROM 0.4 TO 15.0 (Pixels average error)
COLOR_WEIGHT = 0.1         # Reduced slightly to prioritize shape
def resample_contour(contour, n_points=50):
    """Resamples a contour to have exactly n_points using linear interpolation."""
    if len(contour) < 2:
        return np.zeros((n_points, 2), dtype=np.float32)
        
    # Calculate cumulative distance
    dists = np.sqrt(np.sum(np.diff(contour, axis=0)**2, axis=1))
    cum_dists = np.insert(np.cumsum(dists), 0, 0)
    total_len = cum_dists[-1]
    
    if total_len == 0:
        return np.zeros((n_points, 2), dtype=np.float32)
        
    # Interpolate
    x = contour[:, 0]
    y = contour[:, 1]
    target_dists = np.linspace(0, total_len, n_points)
    
    new_x = np.interp(target_dists, cum_dists, x)
    new_y = np.interp(target_dists, cum_dists, y)
    
    return np.column_stack((new_x, new_y)).astype(np.float32)

def get_side_color(image, contour):
    """
    Extracts the average Lab color of the edge.
    We draw the contour as a mask and sample pixels.
    """
    mask = np.zeros(image.shape[:2], dtype=np.uint8)
    # Draw thicker line to capture edge color
    cv2.drawContours(mask, [contour.astype(np.int32)], -1, 255, thickness=4)
    
    # Extract pixels
    pixels = image[mask > 0]
    if len(pixels) == 0:
        return [0, 0, 0]
        
    # Convert to Lab for better color perception matching
    pixels_bgr = pixels.reshape(1, -1, 3).astype(np.uint8)
    pixels_lab = cv2.cvtColor(pixels_bgr, cv2.COLOR_BGR2Lab)
    
    # Return mean L, a, b
    mean_lab = np.mean(pixels_lab[0], axis=0)
    return mean_lab.tolist()

def classify_side(contour):
    """
    Determines if a side is FLAT, OUT (Tab), or IN (Hole).
    Uses signed distance from the straight line connecting corners.
    """
    if len(contour) < 5:
        return "flat", 0.0
        
    start = contour[0]
    end = contour[-1]
    
    # Vector from start to end
    line_vec = end - start
    line_len = np.linalg.norm(line_vec)
    
    if line_len == 0: 
        return "flat", 0.0

    # Rotate line vector 90 degrees to get normal (pointing "in" to the piece center roughly)
    # Assuming CCW winding, Left is In, Right is Out? 
    # Actually, let's use simple max deviation.
    
    # Distance of all points from the line
    # Cross product method
    cross_prod = np.cross(line_vec, contour - start)
    
    # Max deviation
    max_dev = np.max(np.abs(cross_prod)) / line_len
    mean_dev = np.mean(cross_prod) / line_len # Signed mean tells us In or Out
    
    ratio = max_dev / line_len
    
    if ratio < FLAT_THRESHOLD:
        return "flat", ratio
    
    # Determine In vs Out based on winding
    # For standard image coordinates and clockwise order of sides (Top, Right, Bottom, Left):
    # If we walk the contour, "Out" tabs usually deviate to the Left or Right depending on convention.
    # Heuristic: The area of the polygon formed by the contour and the straight line.
    # If area is added (convex), it's a Tab. If subtracted (concave), it's a Hole.
    # We will use the 'mean_dev' sign.
    
    # Note: This sign depends on corner order. We will calibrate later.
    # For now, we distinguish based on magnitude.
    
    return "curved", mean_dev

In [3]:
# # Database
# pieces_db = {}
# match_candidates = {
#     "top": [], "right": [], "bottom": [], "left": []
# }

# # Load Corners
# if not os.path.exists(CORNER_FILE):
#     print(f"ERROR: {CORNER_FILE} not found. Run previous notebook first!")
# else:
#     with open(CORNER_FILE, 'r') as f:
#         all_corners = json.load(f)

#     print(f"Processing {len(all_corners)} pieces...")

#     for img_name, corners in tqdm(all_corners.items()):
#         img_path = os.path.join(IMAGE_DIR, img_name)
#         mask_path = os.path.join(MASK_DIR, os.path.splitext(img_name)[0] + '_mask.png')
        
#         # Fail-Safe: Check files
#         if not os.path.exists(img_path) or not os.path.exists(mask_path):
#             continue
            
#         # Load
#         img = cv2.imread(img_path)
#         mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
#         # Get Contour
#         contours_raw, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
#         if not contours_raw: continue
#         main_contour = max(contours_raw, key=cv2.contourArea).squeeze()
        
#         # Check corners
#         if len(corners) != 4:
#             continue # Skip malformed predictions
            
#         corners = np.array(corners)
        
#         # --- EXTRACT SIDES ---
#         # Snap corners to nearest contour point for precision
#         corner_indices = []
#         for pt in corners:
#             dists = np.sum((main_contour - pt)**2, axis=1)
#             corner_indices.append(np.argmin(dists))
            
#         # Sort indices to follow contour order (0->1->2->3)
#         # But we need to maintain TL, TR, BR, BL logic. 
#         # We assume corners input is sorted TL, TR, BR, BL from previous step.
        
#         sides_data = []
#         side_names = ["top", "right", "bottom", "left"]
        
#         for i in range(4):
#             idx1 = corner_indices[i]
#             idx2 = corner_indices[(i + 1) % 4]
            
#             # Extract points
#             if idx2 > idx1:
#                 side_contour = main_contour[idx1:idx2+1]
#             else:
#                 # Wrap around
#                 side_contour = np.vstack((main_contour[idx1:], main_contour[:idx2+1]))
                
#             # FAIL-SAFE: Check if we went the "long way" around
#             # A side should be roughly 1/4 of perimeter. If it's > 50%, we swapped indices.
#             if len(side_contour) > 0.6 * len(main_contour):
#                 # This usually means corners are out of order, but let's try to reverse the logic?
#                 # Actually, just mark it as invalid to be safe.
#                 side_type = "flat" # Fallback
#                 side_feat = np.zeros((NUM_RESAMPLE_POINTS, 2))
#                 side_color = [0,0,0]
#             else:
#                 # Process Side
#                 resampled = resample_contour(side_contour, NUM_RESAMPLE_POINTS)
                
#                 # Normalize shape for matching (Translation invariant)
#                 # Subtract center
#                 center = np.mean(resampled, axis=0)
#                 norm_shape = resampled - center
                
#                 # Rotate to align start-end with X-axis (Rotation invariant)
#                 start, end = norm_shape[0], norm_shape[-1]
#                 angle = np.arctan2(end[1] - start[1], end[0] - start[0])
#                 c, s = np.cos(-angle), np.sin(-angle)
#                 R = np.array([[c, -s], [s, c]])
#                 norm_shape = np.dot(norm_shape, R.T)
                
#                 # Classify
#                 cls, dev_val = classify_side(side_contour)
                
#                 # Refine Class (Tab vs Hole)
#                 # In our rotated frame, if the middle y is Positive -> Tab, Negative -> Hole
#                 mid_y = norm_shape[len(norm_shape)//2, 1]
                
#                 if cls == "flat":
#                     final_type = "flat"
#                 elif mid_y > 0: # Deviation Up
#                     final_type = "out"
#                 else:
#                     final_type = "in"
                
#                 # Color
#                 color_lab = get_side_color(img, side_contour)
                
#                 side_info = {
#                     "type": final_type,
#                     "shape": norm_shape, # Normalized (50,2)
#                     "color": np.array(color_lab),
#                     "raw_contour": side_contour
#                 }
#                 sides_data.append(side_info)
                
#                 # Add to candidates list for matching
#                 match_candidates[side_names[i]].append({
#                     "img": img_name,
#                     "data": side_info
#                 })
        
#         pieces_db[img_name] = {
#             "sides": dict(zip(side_names, sides_data))
#         }

# print("Database built successfully.")

Processing 1000 pieces...


  0%|          | 0/1000 [00:00<?, ?it/s]

C:\Users\aja21\AppData\Local\Temp\ipykernel_67760\4072165917.py:89: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  cross_prod = np.cross(line_vec, contour - start)


KeyboardInterrupt: 

In [ ]:
# # Helper for matching scores
# def calculate_match_score(side_A, side_B):
#     """
#     Lower score is better.
#     Compares Shape (inverted) and Color.
#     """
#     # 1. Type Check
#     if side_A['type'] == 'flat' and side_B['type'] == 'flat':
#         return 0.0 # Flat matches Flat (Boundary)
#     if side_A['type'] == side_B['type']: 
#         return 100.0 # Tab cannot match Tab, Hole cannot match Hole
#     if side_A['type'] == 'flat' or side_B['type'] == 'flat':
#         return 100.0 # Flat cannot match Curved
        
#     # 2. Shape Matching
#     # Side A is "Out", Side B is "In".
#     # Since we normalized them to the X-axis, "Out" goes Up (+Y), "In" goes Down (-Y).
#     # If they fit perfectly, shape_A + shape_B should be roughly 0 (if we flip B).
#     # Actually, simpler: shape_A should look like flipped shape_B.
    
#     shape_A = side_A['shape']
#     shape_B = side_B['shape']
    
#     # Flip B over X-axis to see if it matches A
#     shape_B_flipped = shape_B.copy()
#     shape_B_flipped[:, 1] *= -1 
    
#     # Euclidean distance between points
#     shape_diff = np.mean(np.linalg.norm(shape_A - shape_B_flipped, axis=1))
    
#     # 3. Color Matching
#     # Colors should be identical across the boundary
#     col_A = side_A['color']
#     col_B = side_B['color']
#     col_diff = np.linalg.norm(col_A - col_B)
    
#     # Normalize color diff (Lab distance ~0-100 range usually)
#     # Normalize shape diff (Pixels, ~0-20 range usually)
    
#     # Weighted Score
#     # Heuristic scaling
#     score = (shape_diff * 1.0) + (col_diff * 0.1 * COLOR_WEIGHT)
    
#     return score

# # --- MAIN MATCHING LOOP ---
# final_matches = {}
# print("Running SOTA Matching Algorithm...")

# adj_graph = {}

# # Opposites
# pairs = [("top", "bottom"), ("right", "left")] # We check Top(A) vs Bottom(B), Right(A) vs Left(B)

# for side_A_name, side_B_name in pairs:
    
#     # Get all candidates
#     candidates_A = match_candidates[side_A_name]
#     candidates_B = match_candidates[side_B_name]
    
#     used_B = set()
    
#     # Iterate through A
#     for entry_A in tqdm(candidates_A, desc=f"Matching {side_A_name}"):
#         img_A = entry_A['img']
#         data_A = entry_A['data']
        
#         # Skip flats
#         if data_A['type'] == 'flat':
#             if img_A not in adj_graph: adj_graph[img_A] = {}
#             adj_graph[img_A][side_A_name] = None
#             continue
            
#         best_match = None
#         best_score = 1000.0
        
#         # Search all B
#         for entry_B in candidates_B:
#             img_B = entry_B['img']
#             if img_B == img_A: continue # Cannot match self
#             if img_B in used_B: continue # Greedy: Already taken (optional, can remove for better global opt)
            
#             data_B = entry_B['data']
            
#             score = calculate_match_score(data_A, data_B)
            
#             if score < best_score:
#                 best_score = score
#                 best_match = img_B
        
#         # Threshold Check
#         if best_score < MATCH_THRESHOLD * 100: # Scale threshold liberally
#             # Store Match
#             if img_A not in adj_graph: adj_graph[img_A] = {}
#             adj_graph[img_A][side_A_name] = best_match
#             used_B.add(best_match)
#         else:
#             if img_A not in adj_graph: adj_graph[img_A] = {}
#             adj_graph[img_A][side_A_name] = None

# # Save Graph
# final_output = {}
# for img, connections in adj_graph.items():
#     final_output[img] = {
#         "matches": connections,
#         "corners": all_corners[img] # Keep corner info for report
#     }

# with open(GRAPH_FILE, 'w') as f:
#     json.dump(final_output, f, indent=2)

# print(f"SUCCESS: Graph saved to {GRAPH_FILE}")

In [4]:
# --- DIAGNOSTIC MATCHING & VISUALIZATION ---

print(f"--- SOTA DIAGNOSTICS ---")

# 1. Check Side Classification Distribution
type_counts = {"flat": 0, "out": 0, "in": 0}
for side_list in match_candidates.values():
    for item in side_list:
        type_counts[item['data']['type']] += 1

print(f"Side Types Found: {type_counts}")
if type_counts['out'] < 10 or type_counts['in'] < 10:
    print("WARNING: Mostly 'Flat' sides detected. Classification logic or Corner positions may need tweaking.")
    print("Try increasing FLAT_THRESHOLD to 0.05 or 0.08.")

# 2. Run Matching with Reporting
print(f"\n--- Running Matching (Threshold: {MATCH_THRESHOLD}) ---")
final_matches = {}
adj_graph = {}
match_counts = 0

pairs = [("top", "bottom"), ("right", "left")] 

for side_A_name, side_B_name in pairs:
    candidates_A = match_candidates[side_A_name]
    candidates_B = match_candidates[side_B_name]
    
    # Log a few scores to debug
    scores_log = []
    
    for entry_A in tqdm(candidates_A, desc=f"Matching {side_A_name}"):
        img_A = entry_A['img']
        data_A = entry_A['data']
        
        if data_A['type'] == 'flat':
            if img_A not in adj_graph: adj_graph[img_A] = {}
            adj_graph[img_A][side_A_name] = None
            continue
            
        best_match = None
        best_score = 1000.0
        
        for entry_B in candidates_B:
            img_B = entry_B['img']
            if img_B == img_A: continue
            
            # Score
            score = calculate_match_score(data_A, entry_B['data'])
            
            if score < best_score:
                best_score = score
                best_match = img_B
        
        # Keep top scores for debugging
        if best_score < 1000:
            scores_log.append(best_score)

        # Threshold Check
        if best_score < MATCH_THRESHOLD: 
            if img_A not in adj_graph: adj_graph[img_A] = {}
            adj_graph[img_A][side_A_name] = best_match
            match_counts += 1
        else:
            if img_A not in adj_graph: adj_graph[img_A] = {}
            adj_graph[img_A][side_A_name] = None

    if len(scores_log) > 0:
        avg_best = sum(scores_log) / len(scores_log)
        print(f"  > Average 'Best Match' score for {side_A_name}: {avg_best:.2f} (Lower is better)")
        print(f"  > Best single match seen: {min(scores_log):.2f}")

print(f"\nTotal Matches Found: {match_counts}")

# 3. Build & Visualize Graph
final_output = {}
for img, connections in adj_graph.items():
    final_output[img] = {
        "matches": connections,
        # Fail-safe: if corners missing in main dict, skip
        "corners": all_corners.get(img, [[0,0]]*4) 
    }

# Save
with open(GRAPH_FILE, 'w') as f:
    json.dump(final_output, f, indent=2)

# Visualize (Robust)
visualize_results()

--- SOTA DIAGNOSTICS ---
Side Types Found: {'flat': 0, 'out': 4, 'in': 0}
Try increasing FLAT_THRESHOLD to 0.05 or 0.08.

--- Running Matching (Threshold: 15.0) ---


Matching top: 0it [00:00, ?it/s]

Matching right:   0%|          | 0/1 [00:00<?, ?it/s]

NameError: name 'calculate_match_score' is not defined